# E791 $D^+\to\pi^-\pi^+\pi^+$ — coefficient closure with MC normalization

This notebook is the Monte Carlo-normalization counterpart of `02_fit_dynamic_parameters.ipynb`. The physics model, injected coefficients, random-start strategy and Minuit configuration are kept the same; only the normalization method changes.

A fixed independent phase-space Monte Carlo sample is generated once and reused throughout the fit, keeping the NLL deterministic.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()


## 1. E791 Fit-2 truth model


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma":   (1.17, 205.7),
    "rho770":  (1.00,   0.0),
    "NR":      (0.48,  57.3),
    "f0_980":  (0.43, 165.0),
    "f2_1270": (0.76,  57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r*np.cos(phase), r*np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}


## 2. Coefficient-only fit model


In [ ]:
truth = {}

def free_coefficient(name):
    x_truth, y_truth = truth_xy[name]
    truth[f"{name}.x"] = float(x_truth)
    truth[f"{name}.y"] = float(y_truth)
    return RealImag(
        Parameter.coefficient(f"{name}.x", 0.0, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", 0.0, owner=name, step=0.01),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

components = [
    Resonance("sigma",   (0,1), coefficients["sigma"],   mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770",  (0,1), coefficients["rho770"],  mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980",  (0,1), coefficients["f0_980"],  mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0,1), coefficients["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0,1), coefficients["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0,1), coefficients["rho1450"], mass=1.4650, width=0.3100, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]
model = DecayModel(channel, components)


## 3. Fixed Monte Carlo normalization sample

`PhaseSpaceMC` uses a uniform proposal in its generation variables and stores the exact phase-space importance weight. The sample is independent of the toy-generation pool and is frozen during minimization.


In [ ]:
N_NORM = 1_000_000
NORM_SEED = 2027
norm = model.generate_phase_space(N_NORM, seed=NORM_SEED)

print(f"normalization MC points : {norm.size:,}")
print(f"normalization seed      : {NORM_SEED}")
print("finite weights          :", bool(jnp.all(jnp.isfinite(norm.weights))))
print("weight min/max         :", float(jnp.min(norm.weights)), float(jnp.max(norm.weights)))


## 4. Generate pseudo-data


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000
pool = model.generate_phase_space(N_POOL, seed=2000)
truth_cache_pool = model.prepare_cache(pool, norm)
truth_intensity, truth_normalization = truth_cache_pool.evaluate(truth)
target_weights = pool.weights * truth_intensity
data = weighted_resample(jax.random.key(791), pool, target_weights, N_DATA, replace=True)

print(f"candidate pool       : {pool.size:,}")
print(f"pseudo-data events  : {data.size:,}")
print("truth normalization :", float(truth_normalization))


In [ ]:
fig, ax = plt.subplots(figsize=(7.5,6.5))
h=ax.hist2d(np.asarray(data.s12),np.asarray(data.s13),bins=110)
fig.colorbar(h[3],ax=ax,label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]"); ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("E791 Fit-2 pseudo-data — MC normalization")
plt.show()


## 5. Likelihood and broad randomized start


In [ ]:
cache = model.prepare_cache(data, norm)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

minimizer = Minimizer(nll, model.parameters, tolerance=1e-4, verbose=2)
START_SEED=314159; START_RANGE=(-2.5,2.5)
rng=np.random.default_rng(START_SEED)
start_values={p.name:float(rng.uniform(*START_RANGE)) for p in model.parameters if not p.fixed}

print(f"{'parameter':16s} {'truth':>11s} {'start':>11s} {'delta':>11s}")
for p in model.parameters:
    if not p.fixed:
        t=truth[p.name]; s=start_values[p.name]
        print(f"{p.name:16s} {t:11.6f} {s:11.6f} {s-t:+11.6f}")
print("NLL(truth) =",float(nll(truth)))
print("NLL(start) =",float(nll(start_values)))


## 6. JAX gradient check


In [ ]:
gradient_check=minimizer.check_gradient(start_values,step_scale=1e-5,print_table=True)


## 7. Perform exactly one fit


In [ ]:
result=minimizer.fit(start_values=start_values,simplex=False,ncall=100000)
fit_values={p.name:float(result.values[p.name]) for p in model.parameters if not p.fixed}
print("valid          =",bool(result.valid))
print("NLL(start)     =",float(nll(start_values)))
print("NLL(truth)     =",float(nll(truth)))
print("NLL(fit)       =",float(result.fval))
print("fit-truth NLL  =",float(result.fval-nll(truth)))
print("EDM            =",float(result.fmin.edm))
print("function calls =",int(result.nfcn))


## 8. Closure table


In [ ]:
rows=[]
print(f"{'parameter':16s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed: continue
    t=float(truth[p.name]); s=float(start_values[p.name]); f=float(result.values[p.name]); e=float(result.errors[p.name]); pull=(f-t)/e
    rows.append((p.name,t,s,f,e,pull))
    print(f"{p.name:16s} {t:10.5f} {s:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")


## 9. Projection before and after the fit


In [ ]:
projection_cache=model.prepare_cache(pool,norm)
def projection(values,bins):
    intensity,_=projection_cache.evaluate(values)
    w=np.asarray(pool.weights*intensity)
    h12,_=np.histogram(np.asarray(pool.s12),bins=bins,weights=w)
    h13,_=np.histogram(np.asarray(pool.s13),bins=bins,weights=w)
    return h12+h13
sdata=np.concatenate([np.asarray(data.s12),np.asarray(data.s13)])
bins=np.linspace(sdata.min(),sdata.max(),110); centers=0.5*(bins[:-1]+bins[1:])
hd,_=np.histogram(sdata,bins=bins); hs=projection(start_values,bins); hf=projection(fit_values,bins); ht=projection(truth,bins)
for h in (hs,hf,ht): h*=hd.sum()/h.sum()
fig,ax=plt.subplots(figsize=(10,5.5))
ax.errorbar(centers,hd,yerr=np.sqrt(np.maximum(hd,1)),fmt=".",label="toy")
ax.step(centers,hs,where="mid",label="start"); ax.step(centers,hf,where="mid",label="fit"); ax.step(centers,ht,where="mid",linestyle="--",label="truth")
ax.legend(); plt.show()
